## Section 2 - Path Configuration

This section defines all project paths from a single `ROOT` variable and creates the output directories used later in the notebook. Every path is managed with `pathlib.Path`, and YOLO26 ONNX export artifacts are isolated inside a dedicated subfolder.


In [3]:
from pathlib import Path
import torch

ROOT = Path.cwd().resolve()
RAW_DATASET_DIR = ROOT / "raw_dataset_balanced"
STRUCTURED_DATASET_DIR = ROOT / "structured_dataset"
RUNS_DIR = ROOT / "runs"
EXPORT_DIR = ROOT / "exported_models" / "yolo26_onnx_end2end"
DATA_YAML_PATH = STRUCTURED_DATASET_DIR / "data.yaml"

CLASS_NAMES = {
    0: "undefected",
    1: "dirt_defect",
    2: "shape_defect",
}
NUM_CLASSES = len(CLASS_NAMES)
RANDOM_SEED = 42
IMAGE_EXT = ".jpg"
LABEL_EXT = ".txt"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

for output_dir in [STRUCTURED_DATASET_DIR, RUNS_DIR, EXPORT_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

assert RAW_DATASET_DIR.exists(), f"Raw dataset directory was not found: {RAW_DATASET_DIR}"

print(f"ROOT: {ROOT}")
print(f"RAW_DATASET_DIR: {RAW_DATASET_DIR}")
print(f"STRUCTURED_DATASET_DIR: {STRUCTURED_DATASET_DIR}")
print(f"RUNS_DIR: {RUNS_DIR}")
print(f"EXPORT_DIR: {EXPORT_DIR}")
print(f"DATA_YAML_PATH: {DATA_YAML_PATH}")
print(f"CLASS_NAMES: {CLASS_NAMES}")
print(f"DEVICE: {DEVICE}")
print("Section 2 completed: project paths and constants are configured.")


ROOT: C:\Users\mohamed\coding\detectx
RAW_DATASET_DIR: C:\Users\mohamed\coding\detectx\raw_dataset_balanced
STRUCTURED_DATASET_DIR: C:\Users\mohamed\coding\detectx\structured_dataset
RUNS_DIR: C:\Users\mohamed\coding\detectx\runs
EXPORT_DIR: C:\Users\mohamed\coding\detectx\exported_models\yolo26_onnx_end2end
DATA_YAML_PATH: C:\Users\mohamed\coding\detectx\structured_dataset\data.yaml
CLASS_NAMES: {0: 'undefected', 1: 'dirt_defect', 2: 'shape_defect'}
Section 2 completed: project paths and constants are configured.


## Section 3 - Dataset Splitting

This section scans the flat raw dataset, keeps only valid image and label pairs, shuffles them with a fixed seed, and copies them into the standard train, validation, and test directory layout expected by Ultralytics.

In [4]:
import random
import shutil

train_images_dir = STRUCTURED_DATASET_DIR / "images" / "train"
val_images_dir = STRUCTURED_DATASET_DIR / "images" / "val"
test_images_dir = STRUCTURED_DATASET_DIR / "images" / "test"
train_labels_dir = STRUCTURED_DATASET_DIR / "labels" / "train"
val_labels_dir = STRUCTURED_DATASET_DIR / "labels" / "val"
test_labels_dir = STRUCTURED_DATASET_DIR / "labels" / "test"

for split_dir in [
    train_images_dir,
    val_images_dir,
    test_images_dir,
    train_labels_dir,
    val_labels_dir,
    test_labels_dir,
]:
    split_dir.mkdir(parents=True, exist_ok=True)

if train_images_dir.exists() and any(train_images_dir.iterdir()):
    existing_counts = {
        split_name: len(list((STRUCTURED_DATASET_DIR / "images" / split_name).glob(f"*{IMAGE_EXT}")))
        for split_name in ("train", "val", "test")
    }
    print(f"Structured dataset already exists at {STRUCTURED_DATASET_DIR}; skipping split and copy.")
    print(
        f"Existing split counts -> train: {existing_counts['train']}, "
        f"val: {existing_counts['val']}, test: {existing_counts['test']}"
    )
else:
    image_paths = sorted(RAW_DATASET_DIR.glob(f"*{IMAGE_EXT}"))
    valid_pairs = []
    for image_path in image_paths:
        label_path = image_path.with_suffix(LABEL_EXT)
        if not label_path.exists():
            print(f"Warning: missing label for {image_path.name}; skipping this image.")
            continue
        valid_pairs.append((image_path, label_path))

    total_pairs = len(valid_pairs)
    print(f"Total valid image-label pairs found: {total_pairs}")

    rng = random.Random(RANDOM_SEED)
    rng.shuffle(valid_pairs)

    train_count = int(total_pairs * 0.80)
    val_count = int(total_pairs * 0.10)
    test_count = total_pairs - train_count - val_count

    split_pairs = {
        "train": valid_pairs[:train_count],
        "val": valid_pairs[train_count:train_count + val_count],
        "test": valid_pairs[train_count + val_count:],
    }

    split_destinations = {
        "train": (train_images_dir, train_labels_dir),
        "val": (val_images_dir, val_labels_dir),
        "test": (test_images_dir, test_labels_dir),
    }

    for split_name, pairs in split_pairs.items():
        image_dest_dir, label_dest_dir = split_destinations[split_name]
        for image_path, label_path in pairs:
            shutil.copy2(image_path, image_dest_dir / image_path.name)
            shutil.copy2(label_path, label_dest_dir / label_path.name)

    final_counts = {
        split_name: len(list((STRUCTURED_DATASET_DIR / "images" / split_name).glob(f"*{IMAGE_EXT}")))
        for split_name in ("train", "val", "test")
    }
    print(
        f"Final split counts -> train: {final_counts['train']}, "
        f"val: {final_counts['val']}, test: {final_counts['test']}"
    )
    assert sum(final_counts.values()) == total_pairs, "Split counts do not sum to the total number of valid pairs."
    assert final_counts["test"] == test_count, "Remainder images were not assigned to the test split as expected."

print("Section 3 completed: the dataset structure is ready for YOLO training.")

Total valid image-label pairs found: 3432
Final split counts -> train: 2745, val: 343, test: 344
Section 3 completed: the dataset structure is ready for YOLO training.


## Section 4 - Create YOLO `data.yaml`

This section writes the dataset configuration file used by Ultralytics. The file points to the structured dataset root, uses relative image folders for each split, and records the class mapping for the three defect labels.

In [5]:
import yaml

data_config = {
    "path": str(STRUCTURED_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}

DATA_YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
with DATA_YAML_PATH.open("w", encoding="utf-8") as yaml_file:
    yaml.safe_dump(data_config, yaml_file, sort_keys=False, allow_unicode=False)

yaml_contents = DATA_YAML_PATH.read_text(encoding="utf-8")
print("data.yaml contents:")
print(yaml_contents)
print(f"Section 4 completed: wrote the dataset config to {DATA_YAML_PATH}")

data.yaml contents:
path: C:\Users\mohamed\coding\detectx\structured_dataset
train: images/train
val: images/val
test: images/test
nc: 3
names:
  0: undefected
  1: dirt_defect
  2: shape_defect

Section 4 completed: wrote the dataset config to C:\Users\mohamed\coding\detectx\structured_dataset\data.yaml


## Section 5 - Train YOLO26n

This section trains a YOLO26 nano detector on the structured dataset using the configured CUDA device when available. The cell is safe to rerun because it will reuse the expected `best.pt` weights file if it is already present.

In [6]:
from ultralytics import YOLO

RUN_NAME = "yolo26n_defects_end2end_local"
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT_PATH = RUN_DIR / "weights" / "best.pt"

if BEST_PT_PATH.exists():
    print(f"Existing trained weights found at {BEST_PT_PATH}; skipping training.")
else:
    model = YOLO("yolo26n.pt")
    train_results = model.train(
        data=str(DATA_YAML_PATH),
        epochs=300,
        imgsz=960,
        batch=16,
        workers=4,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        patience=50,
        amp=True,
        verbose=True,
        exist_ok=True,
    )
    if hasattr(train_results, "save_dir"):
        print(f"Training artifacts saved to: {train_results.save_dir}")

assert BEST_PT_PATH.exists(), f"Expected best weights file was not found: {BEST_PT_PATH}"
print(f"BEST_PT_PATH: {BEST_PT_PATH}")
print("Section 5 completed: training artifacts are available.")

New https://pypi.org/project/ultralytics/8.4.34 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.2  Python-3.12.11 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\mohamed\coding\detectx\structured_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_sca

### Section 5 Fallback - Recover `BEST_PT_PATH` After an Interrupted Session

Run the next cell only if training finished earlier and the current kernel session no longer has `BEST_PT_PATH` defined. It reconstructs the expected location of the best weights file and confirms that the file exists before later steps continue.

In [ ]:
RUN_NAME = "yolo26n_defects_end2end_local"
BEST_PT_PATH = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
assert BEST_PT_PATH.exists(), f"Expected best weights file does not exist yet: {BEST_PT_PATH}"
print(f"Recovered BEST_PT_PATH: {BEST_PT_PATH}")
print("Section 5 fallback completed: BEST_PT_PATH is restored for downstream cells.")

## Section 6 - Evaluate on the Test Set

This section loads the best trained weights and runs evaluation on the held-out test split using YOLO26's native end-to-end behavior. It prints the headline detection metrics so you can quickly confirm how well the trained model generalizes.

In [7]:
from ultralytics import YOLO

eval_model = YOLO(str(BEST_PT_PATH))
metrics = eval_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=960,
    device=DEVICE,
    verbose=True,
)

results_dict = getattr(metrics, "results_dict", {})
box_metrics = getattr(metrics, "box", None)
map50 = results_dict.get("metrics/mAP50(B)", getattr(box_metrics, "map50", float("nan")))
map50_95 = results_dict.get("metrics/mAP50-95(B)", getattr(box_metrics, "map", float("nan")))
precision = results_dict.get("metrics/precision(B)", getattr(box_metrics, "mp", float("nan")))
recall = results_dict.get("metrics/recall(B)", getattr(box_metrics, "mr", float("nan")))

print(f"mAP50: {float(map50):.4f}")
print(f"mAP50-95: {float(map50_95):.4f}")
print(f"Precision: {float(precision):.4f}")
print(f"Recall: {float(recall):.4f}")
print("Section 6 completed: test-set evaluation finished.")

Ultralytics 8.4.2  Python-3.12.11 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
YOLO26n summary (fused): 122 layers, 2,375,421 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 36.95.4 MB/s, size: 90.8 KB)
val: Scanning C:\Users\mohamed\coding\detectx\structured_dataset\labels\test... 344 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 344/344 1.6Kit/s 0.2s<0.0s
val: New cache created: C:\Users\mohamed\coding\detectx\structured_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 10.9it/s 2.0s0.2s
                   all        344        344      0.865      0.851      0.899      0.859
            undefected        152        152      0.816      0.901      0.882      0.852
           dirt_defect         79         79      0.923      0.909      0.952      0.907
          shape_defect        113        113      0.857      0.743      0.863      0.817


## Section 7 - Export Directly to ONNX

This section exports the trained detector straight to ONNX using a `640` input size for Raspberry Pi deployment. The export intentionally keeps YOLO26's native end-to-end behavior enabled, so the deployed model produces final detections directly in `(1, 300, 6)` form without a separate NMS step.


In [8]:
import shutil
from pathlib import Path

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
existing_onnx = next(iter(sorted(EXPORT_DIR.rglob("*.onnx"))), None)
if existing_onnx is not None:
    ONNX_PATH = existing_onnx
    print(f"Existing ONNX export found at {ONNX_PATH}; skipping export.")
else:
    export_model = YOLO(str(BEST_PT_PATH))
    export_result = export_model.export(
        format="onnx",
        imgsz=640,
    )
    export_result_path = Path(str(export_result))
    candidate_paths = []
    if export_result_path.exists() and export_result_path.is_file() and export_result_path.suffix == ".onnx":
        candidate_paths.append(export_result_path)
    search_root = export_result_path if export_result_path.exists() and export_result_path.is_dir() else export_result_path.parent
    if search_root.exists():
        candidate_paths.extend(sorted(search_root.rglob("*.onnx")))
    candidate_paths = [path for path in candidate_paths if path.exists()]
    assert candidate_paths, f"No ONNX file was found after export. Raw export result: {export_result}"

    source_onnx_path = candidate_paths[0]
    ONNX_PATH = EXPORT_DIR / source_onnx_path.name
    shutil.copy2(source_onnx_path, ONNX_PATH)

assert ONNX_PATH.exists(), f"Expected ONNX file was not found: {ONNX_PATH}"
onnx_size_mb = ONNX_PATH.stat().st_size / (1024 ** 2)
print(f"ONNX_PATH: {ONNX_PATH}")
print(f"ONNX file size: {onnx_size_mb:.2f} MB")
print("Section 7 completed: the ONNX export is available.")


Ultralytics 8.4.2  Python-3.12.11 torch-2.11.0+cu128 CPU (Intel Core i7-14700F)
YOLO26n summary (fused): 122 layers, 2,375,421 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'C:\Users\mohamed\coding\detectx\runs\yolo26n_defects_end2end_local\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.2 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command 'pip install --no-cache-dir "onnxruntime-gpu" ' returned non-zero exit status 1.
WARNING Retry 2/2 failed: Command 'pip install --no-cache-dir "onnxruntime-gpu" ' returned non-zero exit status 1.
WARNING requirements:  Command 'pip install --no-cache-dir "onnxruntime-gpu" ' returned non-zero exit status 1.
   ---------------------------------------- 0.0/207.2 MB ? eta -:--:--
   -- ------------------------------------- 10.7/207.2 MB 55.8 MB/s eta 0:00:04
   ---- ----------------------------------- 22.0/207.2 MB 55.

c:\Users\mohamed\coding\detectx\yolo26_env\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(


ONNX: slimming with onnxslim 0.1.90...


c:\Users\mohamed\coding\detectx\yolo26_env\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 22 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: export success  17.8s, saved as 'C:\Users\mohamed\coding\detectx\runs\yolo26n_defects_end2end_local\weights\best.onnx' (9.4 MB)

Export complete (18.0s)
Results saved to C:\Users\mohamed\coding\detectx\runs\yolo26n_defects_end2end_local\weights
Predict:         yolo predict task=detect model=C:\Users\mohamed\coding\detectx\runs\yolo26n_defects_end2end_local\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\mohamed\coding\detectx\runs\yolo26n_defects_end2end_local\weights\best.onnx imgsz=640 data=C:\Users\mohamed\coding\detectx\structured_dataset\data.yaml  
Visualize:       https://netron.app
ONNX_PATH: C:\Users\mohamed\coding\detectx\exported_models\yolo26_onnx_end2end\best.onnx
ONNX file size: 9.35 MB
Section 7 completed: the ONNX export is available.


## Section 8 - ONNX Runtime Sanity Inference Test

This section loads the exported ONNX model with ONNX Runtime, prepares one validation image with the same letterbox preprocessing expected at deployment time, runs a single inference pass, and prints a few decoded detections. Native YOLO26 end-to-end export produces final detections directly, so the post-processing here filters by confidence only and does not apply NMS.


In [9]:
import cv2
import numpy as np
import onnxruntime as ort

print(f"ONNX Runtime version for sanity check: {ort.__version__}")
print(f"Available ONNX Runtime providers: {ort.get_available_providers()}")


def letterbox_resize(image_bgr: np.ndarray, new_shape=(640, 640), color=(114, 114, 114)):
    original_height, original_width = image_bgr.shape[:2]
    scale = min(new_shape[0] / original_height, new_shape[1] / original_width)
    resized_width = int(round(original_width * scale))
    resized_height = int(round(original_height * scale))

    resized_image = cv2.resize(image_bgr, (resized_width, resized_height), interpolation=cv2.INTER_LINEAR)
    pad_width = new_shape[1] - resized_width
    pad_height = new_shape[0] - resized_height
    pad_left = int(round(pad_width / 2 - 0.1))
    pad_right = int(round(pad_width / 2 + 0.1))
    pad_top = int(round(pad_height / 2 - 0.1))
    pad_bottom = int(round(pad_height / 2 + 0.1))

    padded_image = cv2.copyMakeBorder(
        resized_image,
        pad_top,
        pad_bottom,
        pad_left,
        pad_right,
        cv2.BORDER_CONSTANT,
        value=color,
    )
    return padded_image, scale, (pad_left, pad_top)


session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
input_details = session.get_inputs()
output_details = session.get_outputs()

print(f"Session providers: {session.get_providers()}")
print("Input tensor details:")
for detail in input_details:
    print(f"name={detail.name}, shape={detail.shape}, type={detail.type}")

print("Output tensor details:")
for detail in output_details:
    print(f"name={detail.name}, shape={detail.shape}, type={detail.type}")

val_image_paths = sorted((STRUCTURED_DATASET_DIR / "images" / "val").glob(f"*{IMAGE_EXT}"))
assert val_image_paths, "No validation images were found for the ONNX sanity check."
sample_image_path = val_image_paths[0]
sample_bgr = cv2.imread(str(sample_image_path))
assert sample_bgr is not None, f"Failed to read validation image: {sample_image_path}"

letterboxed_bgr, resize_scale, padding = letterbox_resize(sample_bgr, new_shape=(640, 640), color=(114, 114, 114))
sample_rgb = cv2.cvtColor(letterboxed_bgr, cv2.COLOR_BGR2RGB)
input_tensor = sample_rgb.astype(np.float32) / 255.0
input_tensor = np.transpose(input_tensor, (2, 0, 1))
input_tensor = np.expand_dims(input_tensor, axis=0)

print(f"Sample validation image: {sample_image_path.name}")
print(f"Letterbox resize scale: {resize_scale:.6f}, padding: {padding}")
print(f"Prepared ONNX tensor shape: {input_tensor.shape}")

raw_output = session.run(None, {input_details[0].name: input_tensor})[0]
print(f"Raw output tensor shape: {raw_output.shape}")
print("Expected native YOLO26 end-to-end layout for detection is approximately (1, 300, 6).")

detections = raw_output.astype(np.float32)
if detections.ndim == 3 and detections.shape[0] == 1:
    detections = detections[0]
if detections.ndim != 2:
    raise AssertionError(f"Unexpected detection output rank: {detections.ndim}. Raw shape: {raw_output.shape}")
if detections.shape[1] != 6 and detections.shape[0] == 6:
    detections = detections.T
if detections.shape[1] != 6:
    raise AssertionError(
        f"Unexpected detection output layout: {detections.shape}. "
        "Native YOLO26 end-to-end detection is expected to produce rows of length 6."
    )

confidence_threshold = 0.25
scores = detections[:, 4]
keep_mask = scores >= confidence_threshold
filtered_detections = detections[keep_mask]
print(f"Final detections above confidence {confidence_threshold}: {int(keep_mask.sum())}")
print("No manual NMS is applied here because the end-to-end export already produces final detections.")

if filtered_detections.size:
    top_indices = np.argsort(filtered_detections[:, 4])[::-1][:5]
    for rank, idx in enumerate(top_indices, start=1):
        x1, y1, x2, y2, score, class_id_value = filtered_detections[idx]
        class_id = int(np.clip(np.rint(class_id_value), 0, NUM_CLASSES - 1))
        bbox_xyxy = [float(x1), float(y1), float(x2), float(y2)]
        print(
            f"Detection {rank}: class={CLASS_NAMES[class_id]}, "
            f"confidence={float(score):.4f}, bbox_xyxy={bbox_xyxy}"
        )
else:
    print("No detections exceeded the confidence threshold in the sample inference.")

print("Section 8 completed: the exported ONNX model passed the sanity inference check.")


ONNX Runtime version for sanity check: 1.24.4
Available ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
Session providers: ['CPUExecutionProvider']
Input tensor details:
name=images, shape=[1, 3, 640, 640], type=tensor(float)
Output tensor details:
name=output0, shape=[1, 300, 6], type=tensor(float)
Sample validation image: 0001.jpg
Letterbox resize scale: 0.666667, padding: (0, 120)
Prepared ONNX tensor shape: (1, 3, 640, 640)
Raw output tensor shape: (1, 300, 6)
Expected native YOLO26 end-to-end layout for detection is approximately (1, 300, 6).
Final detections above confidence 0.25: 1
No manual NMS is applied here because the end-to-end export already produces final detections.
Detection 1: class=shape_defect, confidence=0.9316, bbox_xyxy=[0.485321044921875, 294.4531555175781, 336.13818359375, 520.91259765625]
Section 8 completed: the exported ONNX model passed the sanity inference check.


## Section 9 - Summarize Exported Artifacts

This section copies the trained PyTorch checkpoint into the ONNX export directory for convenience, then lists every file stored under that directory with its size. The goal is to make the final deployment artifacts easy to inspect and transfer.


In [10]:
import shutil

backup_best_path = EXPORT_DIR / BEST_PT_PATH.name
shutil.copy2(BEST_PT_PATH, backup_best_path)
print(f"Copied PyTorch best weights to {backup_best_path}")

artifact_paths = sorted(path for path in EXPORT_DIR.rglob("*") if path.is_file())
assert artifact_paths, f"No artifacts were found in {EXPORT_DIR}"
for artifact_path in artifact_paths:
    artifact_size_mb = artifact_path.stat().st_size / (1024 ** 2)
    print(f"{artifact_path.relative_to(EXPORT_DIR)} -> {artifact_size_mb:.2f} MB")

print(f"ONNX model ready for Raspberry Pi deployment: {ONNX_PATH}")
print("Section 9 completed: exported artifacts were summarized.")


Copied PyTorch best weights to C:\Users\mohamed\coding\detectx\exported_models\yolo26_onnx_end2end\best.pt
best.onnx -> 9.35 MB
best.pt -> 5.18 MB
ONNX model ready for Raspberry Pi deployment: C:\Users\mohamed\coding\detectx\exported_models\yolo26_onnx_end2end\best.onnx
Section 9 completed: exported artifacts were summarized.


## Section 10 - Deploying on Raspberry Pi

Use the following command on the Raspberry Pi to install the runtime dependencies:

```bash
python3 -m pip install onnxruntime opencv-python numpy
```

Apply the same preprocessing used in this notebook before inference: letterbox-resize the source image while preserving aspect ratio, pad to `640 x 640` with a constant value of `114`, convert from BGR to RGB, normalize to the range `0.0` to `1.0`, transpose from `HWC` to `CHW`, and add a batch dimension so the final input tensor shape is `(1, 3, 640, 640)`.

Check the ONNX model input tensor details on the Pi. Feed a `float32` tensor with shape `(1, 3, 640, 640)` that matches the preprocessing shown in the sanity-check cell.

The native YOLO26 end-to-end detection output is expected to be approximately `(1, 300, 6)`, where each row is a final detection in the form `[x1, y1, x2, y2, confidence, class_id]` in `xyxy` format.

Do not apply manual NMS after inference. Native YOLO26 end-to-end export already produces final detections directly, so deployment only needs a confidence threshold.

The class order is:

1. `undefected`
2. `dirt_defect`
3. `shape_defect`
